# Generate wrapped interferograms, coherence, and backscatter (dB) maps and animations using OPERA CSLC-S1

---    

This notebook:
- Searches OPERA CSLC-S1 products for your AOI + date range
- Subsets CSLCs **before download** using opera-utils
- Builds interferograms/coherence and merges bursts
- Exports mosaics and visualizations

**Quick start**
1) Set parameters in the next cell (AOI, date range, pairing)
2) Run cells top-to-bottom
3) Outputs land in `savedir/`

**Key toggles**
- `SAVE_WGS84`: save WGS84 GeoTIFF mosaics when True
- `DOWNLOAD_WITH_releaseOGRESS`: show progress bar for downloads
- `USE_WATER_YEAR`: Oct–Sep calendar layout when True
- `pair_mode` / `t_span`: control IFG pairing (all vs fixed separation)

**Outputs**
- Subset CSLC H5: `savedir/subset_cslc/*.h5`
- Mosaics (IFG/COH, native CRS): `savedir/tifs/merged_ifg_*`, `merged_coh_*`
- WGS84 mosaics: `savedir/tifs/WGS84/merged_ifg_WGS84_*`, `merged_coh_WGS84_*`, `merged_bsc_<mode>_WGS84_*` (backscatter, dB)
- Calibrated backscatter (dB) mosaics (native CRS; <mode> = sigma0 or beta0): `savedir/tifs/merged_bsc_<mode>_*.tif`
- GIFs: `savedir/gifs/*.gif`

### Data Used in the Example:   

- **10 meter (Northing) x 5 meter (Easting) North America OPERA Coregistered Single Look Complex from Sentinel-1 products**
    - This dataset contains Level-2 OPERA coregistered single-look-complex (CSLC) data from Sentinel-1 (S1). <span style="color:red"> The data in this example are geocoded CSLC-S1 data covering Palos Verdes landslides, California, USA</span>. 
    
    - The OPERA project is generating geocoded burst-wise CSLC-S1 products over North America which includes USA and US Territories within 200 km from the US border, Canada, and all mainland countries from the southern US border down to and including Panama. Each pixel within a burst SLC is represented by a complex number and contains both the amplitude and phase information. The CSLC-S1 products are distributed over projected map coordinates using the Universal Transverse Mercator (UTM) projection with spacing in the X- and Y-directions of 5 m and 10 m, respectively. Each OPERA CSLC-S1 product is distributed as a HDF5 file following the CF-1.8 convention with separate groups containing the data raster layers, the low-resolution correction layers, and relevant product metadata.

    - For more information about the OPERA project and other products please visit our website at https://www.jpl.nasa.gov/go/opera .

Please refer to the [OPERA Product Specification Document](https://d2pn8kiwq2w21t.cloudfront.net/documents/OPERA_CSLC-S1_ProductSpec_v1.0.0_D-108278_Initial_2023-09-11_URS321269.pdf) for details about the CSLC-S1 product.

*Prepared by Al Handwerger and M. Grace Bato*

---

## 0. Setup your conda environment

Assuming you have conda installed. Open your terminal and run the following:
```

# Create the OPERA CSLC environment
conda (or mamba) env create -f environment.yml
conda (or mamba) activate opera_cslc_slides
python -m ipykernel install --user --name opera_cslc_slides

```

---

## 1. Load Python modules

In [ ]:
## Load necessary modules
%load_ext watermark

import asf_search as asf
import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import datetime as dt
import folium
import geopandas as gpd
import glob
import h5py
import imageio.v2 as imageio
import math
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from numpy.typing import NDArray
import os
import pandas as pd
from pathlib import Path
from pyproj import Transformer
import rasterio
from rasterio import merge
from rasterio.crs import CRS
from rasterio.io import MemoryFile
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling
import requests
import re
import rioxarray
from shapely.geometry import box, Point
from shapely.ops import transform as shp_transform
import shapely
import shapely.wkt as wkt
from subprocess import Popen
from platform import system
import sys
import tempfile
import time
from tqdm.auto import tqdm
import warnings

from affine import Affine
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from netrc import netrc
from opera_utils.credentials import get_earthdata_username_password
from opera_utils.disp._remote import open_file
from opera_utils.disp._utils import _get_netcdf_encoding
from osgeo import gdal

proj_dir = os.path.join(sys.prefix, "share", "proj")
os.environ["PROJ_LIB"] = proj_dir  # for older PROJ
os.environ["PROJ_DATA"] = proj_dir  # for newer PROJ

%watermark --iversions


import seaborn as sns
# ROCKET_CMAP = sns.color_palette('rocket', as_cmap=True)
from scipy.signal import convolve
from scipy.ndimage import distance_transform_edt


In [ ]:
# Environment check
import sys
import importlib

REQUIRED_PKGS = [
    'asf_search','cartopy','folium','geopandas','h5py','imageio','matplotlib','numpy','pandas',
    'pyproj','rasterio','rioxarray','shapely','xarray','opera_utils','tqdm','rich','cmcrameri','seaborn'
]
missing = []
for pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    raise ImportError("Missing packages: " + ', '.join(missing) + ". Activate opera_cslc env or install from environment.yml")

print(f"Python: {sys.executable}")
# Colormap sanity check
import cmcrameri.cm as cmc
# _ = ROCKET_CMAP

print('Environment check OK')

In [ ]:
## Notebook display setup
%matplotlib inline
%config InlineBackend.figure_format='retina'

# Pandas display
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Optional reprojection helper for display

# Avoid lots of warnings printing to notebook from asf_search
warnings.filterwarnings('ignore')

## 2. Set up your NASA Earthdata Login Credentials

In [ ]:
urs = 'urs.earthdata.nasa.gov'
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

netrc_name = "_netrc" if system() == "Windows" else ".netrc"
netrc_path = os.path.expanduser(f"~/{netrc_name}")

def write_netrc():
    username = getpass(prompt=prompts[0])
    password = getpass(prompt=prompts[1])
    with open(netrc_path, 'a') as f:
        f.write(f"\nmachine {urs}\n")
        f.write(f"login {username}\n")
        f.write(f"password {password}\n")
    os.chmod(netrc_path, 0o600)

def has_urs_credentials():
    try:
        creds = netrc(netrc_path).authenticators(urs)
        return creds is not None
    except (FileNotFoundError, NetrcParseError):
        return False

if not has_urs_credentials():
    if not os.path.exists(netrc_path):
        open(netrc_path, 'w').close()
    write_netrc()

os.environ["GDAL_HTTP_NETRC"] = "YES"
os.environ["GDAL_HTTP_NETRC_FILE"] = netrc_path

## 3. Enter user-defined parameters

**Parameter guide (impact on downstream steps):**
- **AOI / orbit / path / burst filters**: control which CSLCs are found and subset; changing these changes *all* downstream data.
- **dateStart / dateEnd**: controls query window and calendar/GIF coverage.
- **pair_mode / pair_t_span_days**: controls which interferometric pairs are formed; impacts IFG/COH density.
- **MULTILOOK / TARGET_PIXEL_M**: sets spatial averaging; affects resolution and noise level in IFG/COH/BSC.
- **CALIBRATION_MODE**: choose backscatter calibration: `sigma0` or `beta0`.
- **COH_METHOD**: `standard` (default) or `phase_only` (previous behavior).
- **COH_APPLY_LEE**: apply Lee filter to coherence (optional smoothing).
- **IFG_APPLY_FILTER**: apply Goldstein filter to interferogram phase.
- **Note**: `IFG_APPLY_FILTER` only affects the **interferogram phase** (for display/plots). Coherence is always computed from the unfiltered data; use `COH_METHOD` to choose standard vs phase-only, and `COH_APPLY_LEE` for optional smoothing.
- **MERGE_BLEND_OVERLAP**: if `True`, blend overlap regions between bursts using a distance‑based feather to reduce seams.
- **MERGE_BLEND_EPS**: small constant to avoid divide‑by‑zero in blend weights. Increase slightly if you see artifacts.
- **APPLY_WATER_MASK / WATER_MASK_PATH**: masks out water before saving outputs.
- **SAVE_WGS84**: optional WGS84 GeoTIFFs for easy display.
- **SKIP_EXISTING_TIFS**: reuses existing GeoTIFFs to speed re-runs (delete old files to force regeneration).



In [ ]:
# User parameters (edit these)
# AOI is a WKT polygon in EPSG:4326
# check ASF Vertex to look for correct pass/path for your AOI https://search.asf.alaska.edu/#/?maxResults=250&dataset=OPERA-S1&polygon=POLYGON((-118.3955%2033.7342,-118.3464%2033.7342,-118.3464%2033.7616,-118.3955%2033.7616,-118.3955%2033.7342))&productTypes=CSLC&resultsLoaded=true&granule=OPERA_L2_CSLC-S1_T071-151230-IW3_20260128T135247Z_20260129T074245Z_S1A_VV_v1.1&zoom=10.065&center=-118.699,33.446
## Enter user-defined parameters
SITE_NAME = "Palos_Verdes_Landslides"  # used for output folder naming
aoi = "POLYGON((-118.3955 33.7342,-118.3464 33.7342,-118.3464 33.7616,-118.3955 33.7616,-118.3955 33.7342))"
orbitPass = "DESCENDING" # ASCENDING or DESCENDING
pathNumber = 71 #71 DESC and 64 ASC
# Optional burst selection before download
# Use subswath (e.g., 'IW2', 'IW3') or specific OPERA burst ID (e.g., 'T071_151230_IW3')
BURST_SUBSWATH = 'IW3'  # e.g., 'IW2' or ['IW2', 'IW3'] or None  
# BURST_SUBSWATH = 'IW3'  # e.g., 'IW2' or ['IW2', 'IW3'] or None  

BURST_ID = None        # e.g., 'T071_151230_IW3' or list of burst IDs

dateStart = dt.datetime.fromisoformat('2025-05-01 00:00:00')         #'YYYY-MM-DD HH:MM:SS'
dateEnd = dt.datetime.fromisoformat('2025-06-01 23:59:59')           #'YYYY-MM-DD HH:MM:SS'

# Pairing options
pair_mode = 't_span'   # 'all' or 't_span'
pair_t_span_days = [12]        # int or list of ints (e.g., [6, 12, 24])

# Seasonal filters (exclude pairs if either endpoint falls in these windows)
EXCLUDE_DATE_RANGES = []  # list of (start, end) like [("2023-12-15","2024-03-31")] or []
EXCLUDE_MONTHDAY_RANGES = [] # list of ("MM-DD","MM-DD") e.g., [("12-15","02-15")], or []

#backscatter
CALIBRATION_MODE = 'sigma0'  # 'sigma0' or 'beta0'
CALIBRATION_FACTOR_IS_AMPLITUDE = True  # False: LUT applies to power (default); True: LUT applies to amplitude, so use squared factor

# Multilooking (spatial averaging)
# Set either MULTILOOK (looks_y, looks_x) OR TARGET_PIXEL_M (meters). TARGET overrides MULTILOOK.
MULTILOOK = (1, 1)  # e.g., (3, 6) for 30m from (dy=10m, dx=5m)
TARGET_PIXEL_M = None  # e.g., 30.0 meter or 90.0 meter or None
GOLDSTEIN_ALPHA = 0.5 # 0 (none) to 1 (strong)
COH_METHOD = 'standard'  # 'standard' or 'phase_only'
COH_USE_GAMMA_NORM = True  # True to apply gamma normalization in plots (coherence only)
COH_NORM_GAMMA = 0.5
COH_KERNEL = 'boxcar'  # 'boxcar' or 'weighted'
COH_KERNEL_NUM_CONV = 3  # only used for weighted kernel
COH_APPLY_LEE = False  # apply Lee filter to coherence
IFG_APPLY_FILTER = True  # apply Goldstein filter to interferogram phase

#for burst overlaps
MERGE_BLEND_OVERLAP = True  # feather overlaps to reduce burst overlaps
MERGE_BLEND_EPS = 1e-6

SAVE_WGS84 = False  # set True to save WGS84 GeoTIFF mosaics
SKIP_EXISTING_TIFS = False  # skip writing GeoTIFFs if they already exist

DOWNLOAD_WITH_PROGRESS = True  # set True for per-file progress bar


min_t_span_days = 0         # minimum separation (days)
max_t_span_days = 12     # maximum separation (days) or None
max_pairs_per_burst = None  # int or None
max_pairs_total = None      # int or None

# Calendar settings
USE_WATER_YEAR = True  # True: Oct–Sep, False: Jan–Dec

DOWNLOAD_PROCESSES = min(8, max(2, (os.cpu_count() or 4) // 2))
# DOWNLOAD_BATCH_SIZE = 5


# Normalize name for filesystem (letters/numbers/_/- only)
site_slug = re.sub(r"[^A-Za-z0-9_-]+", "", SITE_NAME)
orbit_code = orbitPass[0].upper()  # 'A' or 'D'
savedir = f'./{site_slug}_{orbit_code}{pathNumber:03d}/'

# Water mask options
# in params cell
WATER_MASK_PATH = f"{savedir}/water_mask/water_mask_esa_wc2021.tif"
APPLY_WATER_MASK = True

**AOI size guidance**

This notebook is tuned for *small* AOIs (e.g., individual landslides). The main opera-utils release includes fixes for small AOI subsetting. Large AOIs can dramatically increase download time, disk usage, and RAM needs. If you use a large polygon, consider:
- Shorter date ranges or fewer bursts
- Coarser `TARGET_PIXEL_M` (more multilooking)
- Tiling the AOI into smaller polygons and mosaicking later

If you see memory errors or very long runtimes, reduce the AOI size or date range.


In [ ]:
# Utilities (used by multiple sections)

def _load_water_mask_match(mask_path, shape, transform, crs):
    if mask_path is None or not Path(mask_path).exists():
        return None
    with rasterio.open(mask_path) as src:
        src_mask = src.read(1)
        if src.crs == crs and src.transform == transform and src_mask.shape == shape:
            return src_mask
        dst = np.zeros(shape, dtype=src_mask.dtype)
        reproject(
            source=src_mask,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.nearest,
        )
        return dst


def _apply_water_mask(arr, mask):
    # mask: 1 = keep land, 0 = water
    return np.where(mask == 0, np.nan, arr)


def _trim_nan_border(arr, transform):
    data = arr[0] if arr.ndim == 3 else arr
    mask = np.isfinite(data) & (data != 0)
    if not mask.any():
        return arr, transform
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    data = data[r0:r1, c0:c1]
    if arr.ndim == 3:
        arr = data[None, ...]
    else:
        arr = data
    new_transform = transform * Affine.translation(c0, r0)
    return arr, new_transform


def _save_mosaic_utm_to_wgs84(out_path, mosaic, transform, epsg):
    dst_crs = 'EPSG:4326'
    dst_transform, width, height = calculate_default_transform(
        f'EPSG:{epsg}', dst_crs, mosaic.shape[2], mosaic.shape[1],
        *rasterio.transform.array_bounds(mosaic.shape[1], mosaic.shape[2], transform)
    )
    dest = np.zeros((1, height, width), dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=dest,
        src_transform=transform,
        src_crs=f'EPSG:{epsg}',
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
    )
    out_meta = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': mosaic.dtype,
        'crs': dst_crs,
        'transform': dst_transform,
    }
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(dest)


In [ ]:
# ESA WorldCover 2021 water mask (GDAL-only)

ESA_WC_GRID_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/esa_worldcover_grid.fgb"
ESA_WC_BASE_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map"
# ESA WorldCover class codes: 80 = Permanent water bodies
ESA_WC_WATER_CLASSES = {80}


def build_worldcover_water_mask(aoi_wkt, out_path, target_res_deg=None):
    # Create a binary land mask from ESA WorldCover (1=land, 0=water).
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return out_path


    aoi_geom = shapely.wkt.loads(aoi_wkt)
    # Load tile grid and select intersecting tiles
    grid = gpd.read_file(ESA_WC_GRID_URL)
    grid = grid.to_crs("EPSG:4326")
    # Find tile id column (varies by grid version)
    tile_col = next((c for c in grid.columns if 'tile' in c.lower()), None)
    if tile_col is None:
        raise RuntimeError(f"No tile column found in grid columns: {list(grid.columns)}")
    tiles = grid[grid.intersects(aoi_geom)][tile_col].tolist()
    if not tiles:
        raise RuntimeError("No WorldCover tiles intersect AOI")

    print(f"Selected tiles: {tiles}")

    tile_urls = [
        f"{ESA_WC_BASE_URL}/ESA_WorldCover_10m_2021_v200_{t}_Map.tif"
        for t in tiles
    ]

    # Quick URL check for first tile
    first_url = tile_urls[0]
    try:
        _ = gdal.Open(first_url)
    except Exception as e:
        raise RuntimeError(f"GDAL cannot open first tile URL: {first_url}\n{e}")

    vrt_path = out_path.with_suffix(".vrt")
    gdal.BuildVRT(str(vrt_path), tile_urls)

    minx, miny, maxx, maxy = aoi_geom.bounds
    warp_kwargs = dict(
        format="GTiff",
        outputBounds=[minx, miny, maxx, maxy],
        multithread=True,
    )
    if target_res_deg is not None:
        warp_kwargs.update(dict(xRes=target_res_deg, yRes=target_res_deg, targetAlignedPixels=True))

    tmp_map = out_path.with_name(out_path.stem + "_map.tif")
    warp_ds = gdal.Warp(str(tmp_map), str(vrt_path), **warp_kwargs)
    if warp_ds is None:
        raise RuntimeError("GDAL Warp returned None. Check network access/URL.")
    warp_ds = None

    with rasterio.open(tmp_map) as src:
        data = src.read(1)
        profile = src.profile

    mask = (~np.isin(data, list(ESA_WC_WATER_CLASSES))).astype("uint8")
    profile.update(dtype="uint8", count=1, nodata=0)

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mask, 1)

    return out_path


if APPLY_WATER_MASK and WATER_MASK_PATH:
    WATER_MASK_PATH = build_worldcover_water_mask(aoi, WATER_MASK_PATH)

## 4. Query OPERA CSLCs using `asf_search`

In [ ]:
## Search for OPERA CSLC data in ASF DAAC
try:
    search_params = dict(
        intersectsWith= aoi,
        dataset='OPERA-S1',
        processingLevel='CSLC',
        flightDirection = orbitPass,
        start=dateStart,
        end=dateEnd)

    ## Return results
    results = asf.search(**search_params)
    print(f"Length of Results: {len(results)}")

except TypeError:
    search_params = dict(
        intersectsWith= aoi.wkt,
        dataset='OPERA-S1',
        processingLevel='CSLC',
        flightDirection = orbitPass,
        start=dateStart,
        end=dateEnd)

    ## Return results
    results = asf.search(**search_params)
    print(f"Length of Results: {len(results)}")

In [ ]:
## Save the results in a geopandas dataframe
gf = gpd.GeoDataFrame.from_features(results.geojson(), crs='EPSG:4326')

## Filter data based on specified track number
gf = gf[gf.pathNumber==pathNumber]
# gf = gf[gf.pgeVersion=="2.1.1"] 

In [ ]:
# Get only relevant metadata
cslc_df = gf[['operaBurstID', 'fileID', 'startTime', 'stopTime', 'url', 'geometry', 'pgeVersion']]
cslc_df['startTime'] = pd.to_datetime(cslc_df.startTime).dt.date
cslc_df['stopTime'] = pd.to_datetime(cslc_df.stopTime).dt.date

# Extract production time from fileID (2nd date token)
def _prod_time_from_fileid(file_id):
    # Example: OPERA_L2_CSLC-S1_..._20221122T161650Z_20240504T081640Z_...
    parts = str(file_id).split('_')
    return parts[5] if len(parts) > 5 else None

cslc_df['productionTime'] = pd.to_datetime(cslc_df['fileID'].apply(_prod_time_from_fileid), format='%Y%m%dT%H%M%SZ', errors='coerce')

# Keep newest duplicate by productionTime (fallback to pgeVersion, stopTime)
cslc_df = cslc_df.sort_values(by=['operaBurstID', 'startTime', 'productionTime', 'pgeVersion', 'stopTime'])
cslc_df = cslc_df.drop_duplicates(subset=['operaBurstID', 'startTime'], keep='last', ignore_index=True)


def _subswath_from_fileid(file_id):
    # Example: ...-IW2_... -> IW2
    m = re.search(r"-IW[1-3]_", str(file_id))
    return m.group(0)[1:4] if m else None

cslc_df['burstSubswath'] = cslc_df['fileID'].apply(_subswath_from_fileid)

# Optional filtering by subswath or specific burst IDs
if BURST_SUBSWATH:
    if isinstance(BURST_SUBSWATH, (list, tuple, set)):
        subswaths = {str(s).upper() for s in BURST_SUBSWATH}
    else:
        subswaths = {str(BURST_SUBSWATH).upper()}
    cslc_df = cslc_df[cslc_df['burstSubswath'].str.upper().isin(subswaths)]

if BURST_ID:
    if isinstance(BURST_ID, (list, tuple, set)):
        burst_ids = {str(b) for b in BURST_ID}
    else:
        burst_ids = {str(BURST_ID)}
    cslc_df = cslc_df[cslc_df['operaBurstID'].isin(burst_ids)]
cslc_df

In [ ]:
# Build AOI geometry
aoi_geom = wkt.loads(aoi)
aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs="EPSG:4326")

# Map center
centroid = aoi_gdf.geometry[0].centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=9, tiles="Esri.WorldImagery")

# Add CSLC footprints
folium.GeoJson(
    data=cslc_df[['operaBurstID','geometry']].set_geometry('geometry').to_crs("EPSG:4326").__geo_interface__,
    name="CSLC footprints",
    style_function=lambda x: {
        "fillColor": "blue",
        "color": "blue",
        "weight": 2,
        "fillOpacity": 0.1,
    },
).add_to(m)

# Add AOI
folium.GeoJson(
    data=aoi_gdf.__geo_interface__,
    name="AOI",
    style_function=lambda x: {
        "fillColor": "red",
        "color": "red",
        "weight": 2,
        "fillOpacity": 0.1,
    },
).add_to(m)

folium.LayerControl().add_to(m)

m

## 5. Download the CSLC-S1 locally

In [ ]:
## Download step skipped: CSLC subsets are streamed via opera-utils
print('Skipping full CSLC downloads; using opera-utils HTTP subsetting.')

# Sort the CSLC-S1 by burstID and date
cslc_df = cslc_df.sort_values(by=["operaBurstID", "startTime"], ignore_index=True)
# cslc_df

In [ ]:
# Enforce date range on dataframe (useful when re-running with narrower dates)
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
cslc_df = cslc_df[(cslc_df['startTime'] >= date_start_day) & (cslc_df['startTime'] <= date_end_day)]
cslc_df = cslc_df.reset_index(drop=True)
cslc_df

## 6. Read each CSLC-S1 and stack them together


In [ ]:
cslc_stack = []; cslc_dates = []; bbox_stack = []; xcoor_stack = []; ycoor_stack = []


subset_dir = f"{savedir}/subset_cslc"
os.makedirs(subset_dir, exist_ok=True)



def _clamp_chunks(chunks, shape):
    # Ensure chunk dims do not exceed data shape
    return tuple(max(1, min(c, s)) for c, s in zip(chunks, shape))

def _extract_subset(input_obj, outpath, rows, cols, chunks=(1,256,256)):
    X0, X1 = (cols.start, cols.stop) if cols is not None else (None, None)
    Y0, Y1 = (rows.start, rows.stop) if rows is not None else (None, None)
    ds = xr.open_dataset(input_obj, engine="h5netcdf", group="data")
    if 'VV' not in ds.data_vars:
        raise ValueError('Source missing VV data')
    subset = ds.isel(y_coordinates=slice(Y0, Y1), x_coordinates=slice(X0, X1))
    # clamp chunks to data shape
    data_shape = subset["VV"].shape
    safe_chunks = _clamp_chunks(chunks, data_shape)
    subset.to_netcdf(
        outpath,
        engine="h5netcdf",
        group="data",
        encoding=_get_netcdf_encoding(subset, chunks=safe_chunks),
    )
    for group in ("metadata", "identification"):
        with h5py.File(input_obj) as hf, h5py.File(outpath, "a") as dest_hf:
            hf.copy(group, dest_hf, name=group)
    with h5py.File(outpath, "a") as hf:
        ctype = h5py.h5t.py_create(np.complex64)
        ctype.commit(hf["/"].id, np.bytes_("complex64"))

def _subset_h5_to_disk(url, aoi_wkt, out_dir):
    outpath = Path(out_dir) / Path(url).name
    if outpath.exists():
        if outpath.stat().st_size < 100 * 1024:
            outpath.unlink()
        else:
            return outpath

    # determine row/col slices by reading coords
    with open_file(url) as in_f:
        ds = xr.open_dataset(in_f, engine="h5netcdf", group="data")
        xcoor = ds["x_coordinates"].values
        ycoor = ds["y_coordinates"].values
        epsg = int(ds["projection"].values)

    aoi_geom = wkt.loads(aoi_wkt)
    if epsg != 4326:
        transformer = Transformer.from_crs('EPSG:4326', f'EPSG:{epsg}', always_xy=True)
        aoi_geom = shp_transform(transformer.transform, aoi_geom)
    minx, miny, maxx, maxy = aoi_geom.bounds
    x_mask = (xcoor >= minx) & (xcoor <= maxx)
    y_mask = (ycoor >= miny) & (ycoor <= maxy)
    if not x_mask.any() or not y_mask.any():
        raise ValueError('AOI does not intersect this CSLC extent')
    ix = np.where(x_mask)[0]
    iy = np.where(y_mask)[0]
    rows = slice(iy.min(), iy.max()+1)
    cols = slice(ix.min(), ix.max()+1)

    if url.startswith('s3://'):
        with open_file(url) as in_f:
            _extract_subset(in_f, outpath, rows, cols)
    else:
        # HTTPS: download to temp then subset
        with tempfile.NamedTemporaryFile(suffix='.h5') as tf:
            if url.startswith('http'):
                session = requests.Session()
                username, password = get_earthdata_username_password()
                session.auth = (username, password)
                resp = session.get(url, stream=True)
                resp.raise_for_status()
                content_type = resp.headers.get('Content-Type', '').lower()
                if 'text/html' in content_type:
                    raise ValueError('Got HTML response instead of HDF5; check Earthdata login/auth')
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        tf.write(chunk)
                tf.flush()
                if tf.tell() < 100 * 1024:
                    raise ValueError('Downloaded file too small; likely auth/redirect issue')
            _extract_subset(tf.name, outpath, rows, cols)
    # validate output has VV; remove tiny/invalid outputs
    try:
        with h5py.File(outpath, 'r') as h5:
            if '/data/VV' not in h5:
                raise ValueError('Subset missing /data/VV')
        if outpath.stat().st_size < 100 * 1024:
            raise ValueError('Subset file too small')
    except Exception:
        if Path(outpath).exists():
            Path(outpath).unlink()
        raise
    return outpath

def _load_subset(file_id, url, start_date):
    try:
        outpath = _subset_h5_to_disk(url, aoi, subset_dir)
    except FileNotFoundError:
        return None  # skip missing products
    # now read subset locally with h5py (fast)
    with h5py.File(outpath, 'r') as h5:
        cslc = h5['/data/VV'][:]
        xcoor = h5['/data/x_coordinates'][:]
        ycoor = h5['/data/y_coordinates'][:]
        dx = int(h5['/data/x_spacing'][()])
        dy = int(h5['/data/y_spacing'][()])
        epsg = int(h5['/data/projection'][()])
        sensing_start = h5['/metadata/processing_information/input_burst_metadata/sensing_start'][()].astype(str)
        sensing_stop = h5['/metadata/processing_information/input_burst_metadata/sensing_stop'][()].astype(str)
        dims = h5['/metadata/processing_information/input_burst_metadata/shape'][:]
        bounding_polygon = h5['/identification/bounding_polygon'][()].astype(str)
        orbit_direction = h5['/identification/orbit_pass_direction'][()].astype(str)
        center_lon, center_lat = h5['/metadata/processing_information/input_burst_metadata/center']
        wavelength = h5['/metadata/processing_information/input_burst_metadata/wavelength'][()].astype(str)
    subset_bbox = [float(xcoor.min()), float(xcoor.max()), float(ycoor.min()), float(ycoor.max())]
    return cslc, xcoor, ycoor, dx, dy, epsg, sensing_start, sensing_stop, dims, bounding_polygon, orbit_direction, center_lon, center_lat, wavelength, subset_bbox

# Subset with progress (parallel)

items = list(zip(cslc_df.fileID, cslc_df.url, cslc_df.startTime))
import xarray as xr
# Diagnostic: check pixel spacing before multilooking
with open_file(items[0][1]) as in_f:
    ds0 = xr.open_dataset(in_f, engine="h5netcdf", group="data")
    dx0 = float(ds0["x_spacing"].values)
    dy0 = float(ds0["y_spacing"].values)
print(f"Pixel spacing (dx, dy) = ({dx0}, {dy0})")

# Derive anisotropic coherence window from TARGET_PIXEL_M or MULTILOOK
def _odd_at_least_one(n):
    n = max(1, int(round(n)))
    return n if n % 2 == 1 else n + 1

# Aim for ~60 m window; if multilook pixel size exceeds 60 m, use that instead
if TARGET_PIXEL_M is not None:
    base_win_m = TARGET_PIXEL_M
else:
    base_win_m = max(abs(dx0) * MULTILOOK[1], abs(dy0) * MULTILOOK[0])

COH_WIN_M = max(60.0, base_win_m)
COH_WIN_X = _odd_at_least_one(COH_WIN_M / abs(dx0))
COH_WIN_Y = _odd_at_least_one(COH_WIN_M / abs(dy0))
print(f"Using anisotropic COH_WIN (Y,X)=({COH_WIN_Y},{COH_WIN_X}) from COH_WIN_M={COH_WIN_M} m")

# Derive MULTILOOK from TARGET_PIXEL_M if provided
if TARGET_PIXEL_M is not None:
    looks_x = max(1, int(round(TARGET_PIXEL_M / abs(dx0))))
    looks_y = max(1, int(round(TARGET_PIXEL_M / abs(dy0))))
    MULTILOOK = (looks_y, looks_x)
    print(f"Using MULTILOOK={MULTILOOK} for TARGET_PIXEL_M={TARGET_PIXEL_M} (dx={dx0}, dy={dy0})")


results = [None] * len(items)
_t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=DOWNLOAD_PROCESSES) as ex:
    futures = {ex.submit(_load_subset, fileID, url, start_date): i for i, (fileID, url, start_date) in enumerate(items)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Subsetting CSLC'):
        i = futures[fut]
        results[i] = fut.result()
_t1 = time.perf_counter()
print(f"Subset/download time: {_t1 - _t0:.1f} s")

valid_idx = [i for i, res in enumerate(results) if res is not None]
if len(valid_idx) != len(results):
    print(f"Skipping {len(results) - len(valid_idx)} failed subsets")
    cslc_df = cslc_df.iloc[valid_idx].reset_index(drop=True)
    results = [results[i] for i in valid_idx]

for (fileID, start_date), res in zip(zip(cslc_df.fileID, cslc_df.startTime), results):
    cslc, xcoor, ycoor, dx, dy, epsg, sensing_start, sensing_stop, dims, bounding_polygon, orbit_direction, center_lon, center_lat, wavelength, subset_bbox = res
    cslc_stack.append(cslc)
    cslc_dates.append(pd.to_datetime(sensing_start).date())
    if subset_bbox is not None:
        bbox = subset_bbox
    else:
        cslc_poly = wkt.loads(bounding_polygon)
        bbox = [cslc_poly.bounds[0], cslc_poly.bounds[2], cslc_poly.bounds[1], cslc_poly.bounds[3]]
    bbox_stack.append(bbox)
    xcoor_stack.append(xcoor)
    ycoor_stack.append(ycoor)

<details>
<summary>7. Generate the interferograms, compute for the coherence, save the files as GeoTiffs</summary>

</details>

In [ ]:
import h5py, os, glob
f = sorted(glob.glob(f"{savedir}/subset_cslc/*.h5"))[0]
print(f, os.path.getsize(f))
with h5py.File(f, "r") as h5:
    print(list(h5["/data"].keys()))


In [ ]:
def colorize(array=[], cmap='RdBu', cmin=[], cmax=[]):
    normed_data = (array - cmin) / (cmax - cmin)    
    cm = plt.cm.get_cmap(cmap)
    return cm(normed_data) 

In [ ]:
def goldstein_filter(ifg_cpx, alpha=0.5, pad=32, edge_trim=16):
    # Goldstein filter with padding + taper + mask to reduce edge effects
    mask = np.isfinite(ifg_cpx)
    data = np.nan_to_num(ifg_cpx, nan=0.0)
    if pad and pad > 0:
        data = np.pad(data, ((pad, pad), (pad, pad)), mode="reflect")
        mask = np.pad(mask, ((pad, pad), (pad, pad)), mode="constant", constant_values=False)
    # Apply 2D Hann window (taper)
    wy = np.hanning(data.shape[0])
    wx = np.hanning(data.shape[1])
    window = wy[:, None] * wx[None, :]
    f = np.fft.fft2(data * window)
    s = np.abs(f)
    s = s / (s.max() + 1e-8)
    f_filt = f * (s ** alpha)
    out = np.fft.ifft2(f_filt)
    if pad and pad > 0:
        out = out[pad:-pad, pad:-pad]
        mask = mask[pad:-pad, pad:-pad]
    # restore NaNs outside valid mask
    out[~mask] = np.nan
    if edge_trim and edge_trim > 0:
        out[:edge_trim, :] = np.nan
        out[-edge_trim:, :] = np.nan
        out[:, :edge_trim] = np.nan
        out[:, -edge_trim:] = np.nan
    return out

In [ ]:
def rasterWrite(outtif,arr,transform,epsg,dtype='float32'):
    #writing geotiff using rasterio
    
    new_dataset = rasterio.open(outtif, 'w', driver='GTiff',
                            height = arr.shape[0], width = arr.shape[1],
                            count=1, dtype=dtype,
                            crs=CRS.from_epsg(epsg),
                            transform=transform,nodata=np.nan)
    new_dataset.write(arr, 1)
    new_dataset.close() 

In [ ]:
## Build date pairs per burstID
cslc_dates = cslc_df[["startTime"]]
burstID = cslc_df.operaBurstID.drop_duplicates(ignore_index=True)
n_unique_burstID = len(burstID)

def _lag_list(lag):
    if lag is None:
        return []
    if isinstance(lag, (list, tuple, set)):
        return sorted({int(x) for x in lag})
    return [int(lag)]

pair_lags = _lag_list(pair_t_span_days)
pair_indices = []  # list of (ref_idx, sec_idx) in cslc_df order

for bid, group in cslc_df.groupby('operaBurstID'):
    group = group.sort_values('startTime')
    idx = group.index.to_list()
    dates = group['startTime'].to_list()

    burst_pairs = []
    if pair_mode == 'all':
        for i in range(len(idx)):
            for j in range(i+1, len(idx)):
                delta = (dates[j] - dates[i]).days
                if delta < min_t_span_days:
                    continue
                if max_t_span_days is not None and delta > max_t_span_days:
                    continue
                burst_pairs.append((idx[i], idx[j]))
    elif pair_mode == 't_span':
        for i in range(len(idx)):
            for j in range(i+1, len(idx)):
                delta = (dates[j] - dates[i]).days
                if delta in pair_lags and delta >= min_t_span_days and (max_t_span_days is None or delta <= max_t_span_days):
                    burst_pairs.append((idx[i], idx[j]))
    else:
        raise ValueError("pair_mode must be 'all' or 't_span'")

    if max_pairs_per_burst is not None:
        burst_pairs = burst_pairs[:int(max_pairs_per_burst)]

    pair_indices.extend(burst_pairs)

if max_pairs_total is not None:
    pair_indices = pair_indices[:int(max_pairs_total)]

# Sort pairs by date, then burstID (so same dates group together)
def _pair_sort_key(pair):
    ref_idx, sec_idx = pair
    ref_date = cslc_dates.iloc[ref_idx].values[0]
    sec_date = cslc_dates.iloc[sec_idx].values[0]
    burst = cslc_df.operaBurstID.iloc[ref_idx]
    return (ref_date, sec_date, burst)
pair_indices = sorted(pair_indices, key=_pair_sort_key)
print(f'Pair count: {len(pair_indices)}')

# Seasonal filtering based on pair endpoints
from datetime import date

# Normalize exclude ranges to date objects
_excl_ranges = []
for s, e in EXCLUDE_DATE_RANGES:
    try:
        s_d = pd.to_datetime(s).date()
        e_d = pd.to_datetime(e).date()
    except Exception:
        continue
    _excl_ranges.append((s_d, e_d))

# Normalize month-day ranges
if EXCLUDE_MONTHDAY_RANGES:
    if len(EXCLUDE_MONTHDAY_RANGES) == 2 and all(isinstance(x, str) for x in EXCLUDE_MONTHDAY_RANGES):
        EXCLUDE_MONTHDAY_RANGES = [(EXCLUDE_MONTHDAY_RANGES[0], EXCLUDE_MONTHDAY_RANGES[1])]

# Filter pairs: exclude if either endpoint falls in excluded months or ranges
filtered_pairs = []
for r, s in pair_indices:
    d1 = pd.to_datetime(cslc_dates.iloc[r].values[0]).date()
    d2 = pd.to_datetime(cslc_dates.iloc[s].values[0]).date()

    # Date-range exclusion
    excluded = False
    for rs, re in _excl_ranges:
        if rs <= d1 <= re or rs <= d2 <= re:
            excluded = True
            break
    if excluded:
        continue

    # Month-day exclusion (recurring each year)
    if EXCLUDE_MONTHDAY_RANGES:
        md1 = (d1.month, d1.day)
        md2 = (d2.month, d2.day)
        for rng in EXCLUDE_MONTHDAY_RANGES:
            if not (isinstance(rng, (list, tuple)) and len(rng) == 2):
                continue
            s_md, e_md = rng
            try:
                s_m, s_d = map(int, s_md.split('-'))
                e_m, e_d = map(int, e_md.split('-'))
            except Exception:
                continue
            start = (s_m, s_d)
            end = (e_m, e_d)
            # handle ranges that wrap year end (e.g., 12-15 to 02-15)
            def _in_range(md):
                if start <= end:
                    return start <= md <= end
                return md >= start or md <= end
            if _in_range(md1) or _in_range(md2):
                excluded = True
                break
        if excluded:
            continue

    filtered_pairs.append((r, s))

pair_indices = filtered_pairs


In [ ]:
def take_looks(arr, row_looks, col_looks, func_type="nanmean", edge_strategy="cutoff"):
    if row_looks == 1 and col_looks == 1:
        return arr
    if arr.ndim != 2:
        raise ValueError("take_looks expects 2D array")
    rows, cols = arr.shape
    if edge_strategy == "cutoff":
        rows = (rows // row_looks) * row_looks
        cols = (cols // col_looks) * col_looks
        arr = arr[:rows, :cols]
    elif edge_strategy == "pad":
        pad_r = (-rows) % row_looks
        pad_c = (-cols) % col_looks
        if pad_r or pad_c:
            arr = np.pad(arr, ((0, pad_r), (0, pad_c)), mode="constant", constant_values=np.nan)
        rows, cols = arr.shape
    else:
        raise ValueError("edge_strategy must be 'cutoff' or 'pad'")

    new_rows = rows // row_looks
    new_cols = cols // col_looks
    func = getattr(np, func_type)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return func(arr.reshape(new_rows, row_looks, new_cols, col_looks), axis=(1, 3))


def _multilook(arr, looks_y=1, looks_x=1):
    return take_looks(arr, looks_y, looks_x, func_type="nanmean", edge_strategy="cutoff")


def _box_mean(arr, win_y, win_x):
    pad_y = win_y // 2
    pad_x = win_x // 2
    arr_p = np.pad(arr, ((pad_y, pad_y), (pad_x, pad_x)), mode='reflect')
    windows = sliding_window_view(arr_p, (win_y, win_x))
    return windows.mean(axis=(-2, -1))

def get_kernel(size_y, size_x, num_conv):
    if not isinstance(num_conv, int):
        raise ValueError('num_conv must be an integer')
    k0 = np.ones((size_y, size_x), dtype=np.float32)
    k = k0
    for i in range(num_conv):
        if i > 3:
            k = convolve(k, k0, mode='same')
        else:
            k = convolve(k, k0)
    k = k / np.sum(k)
    return k.astype(np.float32)


def _weighted_mean(arr, k):
    # supports complex or real arrays
    return convolve(arr, k, mode='same')



def lee_filter(img, win_y=5, win_x=5):
    mean = _box_mean(img, win_y, win_x)
    mean_sq = _box_mean(img**2, win_y, win_x)
    var = mean_sq - mean**2
    noise_var = np.nanmedian(var)
    w = var / (var + noise_var + 1e-8)
    return mean + w * (img - mean)


def goldstein(
    phase: NDArray[np.complex64] | NDArray[np.float64], alpha: float, psize: int = 32
) -> np.ndarray:
    """Apply the Goldstein adaptive filter to the given data."""

    def apply_pspec(data: NDArray[np.complex64]) -> np.ndarray:
        if alpha < 0:
            raise ValueError(f"alpha must be >= 0, got {alpha = }")
        weight = np.power(np.abs(data) ** 2, alpha / 2)
        data = weight * data
        return data

    def make_weight(nxp: int, nyp: int) -> np.ndarray:
        wx = 1.0 - np.abs(np.arange(nxp // 2) - (nxp / 2.0 - 1.0)) / (nxp / 2.0 - 1.0)
        wy = 1.0 - np.abs(np.arange(nyp // 2) - (nyp / 2.0 - 1.0)) / (nyp / 2.0 - 1.0)
        quadrant = np.outer(wy, wx)
        weight = np.block(
            [
                [quadrant, np.flip(quadrant, axis=1)],
                [np.flip(quadrant, axis=0), np.flip(np.flip(quadrant, axis=0), axis=1)],
            ]
        )
        return weight

    def patch_goldstein_filter(
        data: NDArray[np.complex64], weight: NDArray[np.float64], psize: int
    ) -> np.ndarray:
        data = np.fft.fft2(data, s=(psize, psize))
        data = apply_pspec(data)
        data = np.fft.ifft2(data, s=(psize, psize))
        return weight * data

    def apply_goldstein_filter(data: NDArray[np.complex64]) -> np.ndarray:
        empty_mask = np.isnan(data) | (data == 0)
        if np.all(empty_mask):
            return data

        nrows, ncols = data.shape
        step = psize // 2

        pad_top = step
        pad_left = step
        pad_bottom = step + (step - (nrows % step)) % step
        pad_right = step + (step - (ncols % step)) % step
        data_padded = np.pad(
            data, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="reflect"
        )

        out = np.zeros(data_padded.shape, dtype=np.complex64)
        weight_sum = np.zeros(data_padded.shape, dtype=np.float64)
        weight_matrix = make_weight(psize, psize)

        padded_rows, padded_cols = data_padded.shape
        for i in range(0, padded_rows - psize + 1, step):
            for j in range(0, padded_cols - psize + 1, step):
                data_window = data_padded[i : i + psize, j : j + psize]
                filtered_window = patch_goldstein_filter(
                    data_window, weight_matrix, psize
                )
                out[i : i + psize, j : j + psize] += filtered_window
                weight_sum[i : i + psize, j : j + psize] += weight_matrix

        valid = weight_sum > 0
        out[valid] /= weight_sum[valid]

        out = out[pad_top : pad_top + nrows, pad_left : pad_left + ncols]
        out[empty_mask] = 0
        return out

    if np.iscomplexobj(phase):
        return apply_goldstein_filter(phase)
    else:
        return apply_goldstein_filter(np.exp(1j * phase))


def calc_ifg_coh_filtered(reference, secondary, goldstein_alpha=0.5, coh_win_y=5, coh_win_x=5, looks_y=1, looks_x=1, coh_method="standard", ifg_apply_filter=True, coh_apply_lee=False, coh_kernel="boxcar", coh_kernel_num_conv=5):
    reference = _multilook(reference, looks_y, looks_x)
    secondary = _multilook(secondary, looks_y, looks_x)
    phase = reference * np.conjugate(secondary)
    amp = np.sqrt((reference * np.conjugate(reference)) * (secondary * np.conjugate(secondary)))
    nan_mask = np.isnan(phase)
    ifg_cpx = np.exp(1j * np.nan_to_num(np.angle(phase/amp)))
    if ifg_apply_filter:
        ifg_cpx_f = goldstein(ifg_cpx, alpha=goldstein_alpha, psize=32)
    else:
        ifg_cpx_f = ifg_cpx
    ifg = np.angle(ifg_cpx_f)
    ifg[nan_mask] = np.nan

    ifg_cpx_used = ifg_cpx  # coherence never uses filtered IFG

    if coh_kernel == 'weighted':
        k = get_kernel(coh_win_y, coh_win_x, coh_kernel_num_conv)
    elif coh_kernel == 'boxcar':
        k = None
    else:
        raise ValueError("coh_kernel must be 'boxcar' or 'weighted'")
    if coh_method == "phase_only":
        if coh_kernel == 'weighted':
            coh = np.abs(_weighted_mean(ifg_cpx_used, k))
        else:
            coh = np.abs(_box_mean(ifg_cpx_used, coh_win_y, coh_win_x))
    elif coh_method == "standard":
        if coh_kernel == 'weighted':
            num = np.abs(_weighted_mean(phase, k))
            den = np.sqrt(_weighted_mean(np.abs(reference)**2, k) * _weighted_mean(np.abs(secondary)**2, k))
        else:
            num = np.abs(_box_mean(phase, coh_win_y, coh_win_x))
            den = np.sqrt(_box_mean(np.abs(reference)**2, coh_win_y, coh_win_x) * _box_mean(np.abs(secondary)**2, coh_win_y, coh_win_x))
        coh = np.where(den > 0, num / den, 0)
    else:
        raise ValueError("coh_method must be 'standard' or 'phase_only'")
    coh = np.clip(coh, 0, 1)
    if coh_apply_lee:
        coh = lee_filter(coh, win_y=coh_win_y, win_x=coh_win_x)
        coh = np.clip(coh, 0, 1)
    zero_mask = phase == 0
    coh[nan_mask] = np.nan
    coh[zero_mask] = 0
    return ifg, coh, amp


def calc_ifg_coh(reference, secondary, goldstein_alpha=0.5, coh_win_y=5, coh_win_x=5, looks_y=1, looks_x=1, coh_method="standard", ifg_apply_filter=True, coh_apply_lee=False, coh_kernel="boxcar", coh_kernel_num_conv=5):
    return calc_ifg_coh_filtered(reference, secondary, goldstein_alpha=goldstein_alpha, coh_win_y=coh_win_y, coh_win_x=coh_win_x, looks_y=looks_y, looks_x=looks_x, coh_method=coh_method, ifg_apply_filter=ifg_apply_filter, coh_apply_lee=coh_apply_lee, coh_kernel=coh_kernel, coh_kernel_num_conv=coh_kernel_num_conv)


In [ ]:
## For each date-pair, calculate the ifg, coh. Save the results as GeoTiffs.
for ref_idx, sec_idx in pair_indices:
    ref_date = cslc_dates.iloc[ref_idx].values[0]
    sec_date = cslc_dates.iloc[sec_idx].values[0]
    print(f"Reference: {ref_date}  Secondary: {sec_date}")

    # Calculate ifg, coh, amp
    if "calc_ifg_coh_filtered" not in globals():
        raise RuntimeError("calc_ifg_coh_filtered is not defined. Run the filter definition cell first.")
    looks_y, looks_x = MULTILOOK

    # Save each interferogram as GeoTiff (no per-burst plotting)
    transform = from_origin(xcoor_stack[ref_idx][0], ycoor_stack[ref_idx][0], dx, np.abs(dy))

<details>
<summary>8. Merge the burst-wise interferograms and coherence and save as GeoTiff.</summary>

</details>

In [ ]:
def custom_merge(old_data, new_data, old_nodata, new_nodata, **kwargs):
    # Feather overlaps to reduce burst seams
    if MERGE_BLEND_OVERLAP:
        overlap = np.logical_and(~old_nodata, ~new_nodata)
        if np.any(overlap):
            # distance to nodata inside each valid mask
            dist_old = distance_transform_edt(~old_nodata)
            dist_new = distance_transform_edt(~new_nodata)
            w_new = dist_new / (dist_new + dist_old + MERGE_BLEND_EPS)
            w_new = np.clip(w_new, 0, 1)
            blended = old_data * (1 - w_new) + new_data * w_new
            old_data[overlap] = blended[overlap]
    # fill empty pixels
    mask = np.logical_and(old_nodata, ~new_nodata)
    old_data[mask] = new_data[mask]


In [ ]:
os.makedirs(f"{savedir}/tifs", exist_ok=True)
# Merge burst-wise interferograms per date-pair

# Group pair indices by date tag
pairs_by_tag = {}
for r, s in pair_indices:
    ref_date = cslc_dates.iloc[r].values[0]
    sec_date = cslc_dates.iloc[s].values[0]
    tag = f"{ref_date.strftime('%Y%m%d')}-{sec_date.strftime('%Y%m%d')}"
    pairs_by_tag.setdefault(tag, []).append((r, s))

for tag, pairs in pairs_by_tag.items():
    srcs = []
    for r, s in pairs:
        looks_y, looks_x = MULTILOOK
        ifg, coh, amp = calc_ifg_coh(
            cslc_stack[r], cslc_stack[s],
            goldstein_alpha=GOLDSTEIN_ALPHA, coh_win_y=COH_WIN_Y, coh_win_x=COH_WIN_X,
            looks_y=looks_y, looks_x=looks_x,
            coh_method=COH_METHOD,
            ifg_apply_filter=IFG_APPLY_FILTER,
            coh_apply_lee=COH_APPLY_LEE,
            coh_kernel=COH_KERNEL, coh_kernel_num_conv=COH_KERNEL_NUM_CONV,
        )
        dy_signed = (ycoor_stack[r][1] - ycoor_stack[r][0]) if len(ycoor_stack[r]) > 1 else -dy
        x0 = xcoor_stack[r][0] + (looks_x - 1) * dx / 2
        y0 = ycoor_stack[r][0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=ifg.shape[0], width=ifg.shape[1], count=1, dtype=ifg.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(ifg, 1)
        srcs.append(ds)
    dest, output_transform = merge.merge(srcs, method=custom_merge)
    dest, output_transform = _trim_nan_border(dest, output_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, dest.shape[1:], output_transform, CRS.from_epsg(epsg))
        if mask is not None:
            dest[0] = _apply_water_mask(dest[0], mask)
    out_meta = srcs[0].meta.copy()
    out_meta.update({"driver": "GTiff", "height": dest.shape[1], "width": dest.shape[2], "transform": output_transform})
    out_path = f"{savedir}/tifs/merged_ifg_{tag}.tif"
    if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path)):
        with rasterio.open(out_path, "w", **out_meta) as dest1:
            dest1.write(dest)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_ifg_WGS84_{tag}.tif"
        if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path_wgs84)):
            _save_mosaic_utm_to_wgs84(out_path_wgs84, dest, output_transform, epsg)
    for ds in srcs:
        ds.close()


In [ ]:
# Merge burst-wise coherence per date-pair

# Group pair indices by date tag
pairs_by_tag = {}
for r, s in pair_indices:
    ref_date = cslc_dates.iloc[r].values[0]
    sec_date = cslc_dates.iloc[s].values[0]
    tag = f"{ref_date.strftime('%Y%m%d')}-{sec_date.strftime('%Y%m%d')}"
    pairs_by_tag.setdefault(tag, []).append((r, s))

for tag, pairs in pairs_by_tag.items():
    srcs = []
    for r, s in pairs:
        looks_y, looks_x = MULTILOOK
        ifg, coh, amp = calc_ifg_coh(
            cslc_stack[r], cslc_stack[s],
            goldstein_alpha=GOLDSTEIN_ALPHA, coh_win_y=COH_WIN_Y, coh_win_x=COH_WIN_X,
            looks_y=looks_y, looks_x=looks_x,
            coh_method=COH_METHOD,
            ifg_apply_filter=IFG_APPLY_FILTER,
            coh_apply_lee=COH_APPLY_LEE,
            coh_kernel=COH_KERNEL, coh_kernel_num_conv=COH_KERNEL_NUM_CONV,
        )
        dy_signed = (ycoor_stack[r][1] - ycoor_stack[r][0]) if len(ycoor_stack[r]) > 1 else -dy
        x0 = xcoor_stack[r][0] + (looks_x - 1) * dx / 2
        y0 = ycoor_stack[r][0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=coh.shape[0], width=coh.shape[1], count=1, dtype=coh.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(coh, 1)
        srcs.append(ds)
    dest, output_transform = merge.merge(srcs, method=custom_merge)
    dest, output_transform = _trim_nan_border(dest, output_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, dest.shape[1:], output_transform, CRS.from_epsg(epsg))
        if mask is not None:
            dest[0] = _apply_water_mask(dest[0], mask)
    out_meta = srcs[0].meta.copy()
    out_meta.update({"driver": "GTiff", "height": dest.shape[1], "width": dest.shape[2], "transform": output_transform})
    out_path = f"{savedir}/tifs/merged_coh_{tag}.tif"
    if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path)):
        with rasterio.open(out_path, "w", **out_meta) as dest1:
            dest1.write(dest)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_coh_WGS84_{tag}.tif"
        if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path_wgs84)):
            _save_mosaic_utm_to_wgs84(out_path_wgs84, dest, output_transform, epsg)
    for ds in srcs:
        ds.close()


<details>
<summary>9. Read the merged GeoTiff and Visualize using `matplotlib`</summary>

</details>

In [ ]:
# Read merged IFG/COH files and plot paired grids


# Output dir for per-pair PNGs
pair_png_dir = f"{savedir}/pairs_png"
os.makedirs(pair_png_dir, exist_ok=True)

ifg_paths = sorted(glob.glob(f"{savedir}/tifs/merged_ifg_*.tif"))
coh_norm = mcolors.PowerNorm(gamma=COH_NORM_GAMMA, vmin=0, vmax=1) if COH_USE_GAMMA_NORM else None
coh_paths = sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif"))

ifg_map = {p.split('merged_ifg_')[-1].replace('.tif',''): p for p in ifg_paths}
coh_map = {p.split('merged_coh_')[-1].replace('.tif',''): p for p in coh_paths}



def _prep_da(path):
    da = rioxarray.open_rasterio(path)[0]
    data = da.values
    mask = np.isfinite(data) & (data != 0)
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        r0, r1 = rows[0], rows[-1] + 1
        c0, c1 = cols[0], cols[-1] + 1
        # Trim NaN borders so edges don't show padding
        da = da.isel(y=slice(r0, r1), x=slice(c0, c1))
    return da

pair_tags = sorted(set(ifg_map).intersection(coh_map))
# Filter pairs by current date range
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
pair_tags = [t for t in pair_tags if (date_start_day <= pd.to_datetime(t.split('-')[0], format='%Y%m%d').date() <= date_end_day and date_start_day <= pd.to_datetime(t.split('-')[1], format='%Y%m%d').date() <= date_end_day)]

if not pair_tags:
    print('No matching IFG/COH pairs found')
else:
    # Save ALL pairs as PNGs
    for tag in pair_tags:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        ax_ifg, ax_coh = axes

        # IFG
        merged_ifg = _prep_da(ifg_map[tag])
        minlon, minlat, maxlon, maxlat = merged_ifg.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        colored_ifg = colorize(merged_ifg, 'twilight_shifted', -np.pi, np.pi)
        colored_ifg = np.ma.masked_invalid(colored_ifg)
        im_ifg = ax_ifg.imshow(colored_ifg, cmap='twilight_shifted', interpolation='none', origin='upper', extent=bbox, vmin=-np.pi, vmax=np.pi)
        ax_ifg.set_title(f"IFG_{tag}", fontsize=10)
        ax_ifg.set_xticks([])
        ax_ifg.set_yticks([])
        fig.colorbar(im_ifg, ax=ax_ifg, orientation='vertical', fraction=0.046, pad=0.02, label='Wrapped phase (rad)')

        # COH
        merged_coh = _prep_da(coh_map[tag])
        minlon, minlat, maxlon, maxlat = merged_coh.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        coh_vals = np.ma.masked_invalid(merged_coh.values)
        im_coh = ax_coh.imshow(coh_vals, cmap='gray', interpolation='none', origin='upper', extent=bbox, norm=coh_norm, vmin=None if COH_USE_GAMMA_NORM else 0, vmax=None if COH_USE_GAMMA_NORM else 1.0)
        ax_coh.set_title(f"COH_{tag}", fontsize=10)
        ax_coh.set_xticks([])
        ax_coh.set_yticks([])
        fig.colorbar(im_coh, ax=ax_coh, orientation='vertical', fraction=0.046, pad=0.02, label='Coherence')

        out_png = os.path.join(pair_png_dir, f"pair_{tag}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    # Display only last 5 pairs in notebook
    display_tags = pair_tags[-5:]
    n = len(display_tags)
    ncols = 2
    nrows = math.ceil(n / 1)  # one pair per row
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 3*nrows), constrained_layout=True)
    if nrows == 1:
        axes = [axes]

    for i, tag in enumerate(display_tags):
        ax_ifg, ax_coh = axes[i]

        # IFG
        merged_ifg = _prep_da(ifg_map[tag])
        minlon, minlat, maxlon, maxlat = merged_ifg.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        colored_ifg = colorize(merged_ifg, 'twilight_shifted', -np.pi, np.pi)
        colored_ifg = np.ma.masked_invalid(colored_ifg)
        im_ifg = ax_ifg.imshow(colored_ifg, cmap='twilight_shifted', interpolation='none', origin='upper', extent=bbox, vmin=-np.pi, vmax=np.pi)
        ax_ifg.set_title(f"IFG_{tag}", fontsize=10)
        ax_ifg.set_xticks([])
        ax_ifg.set_yticks([])
        fig.colorbar(im_ifg, ax=ax_ifg, orientation='vertical', fraction=0.046, pad=0.02, label='Wrapped phase (rad)')

        # COH
        merged_coh = _prep_da(coh_map[tag])
        minlon, minlat, maxlon, maxlat = merged_coh.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        coh_vals = np.ma.masked_invalid(merged_coh.values)
        im_coh = ax_coh.imshow(coh_vals, cmap='gray', interpolation='none', origin='upper', extent=bbox, norm=coh_norm, vmin=None if COH_USE_GAMMA_NORM else 0, vmax=None if COH_USE_GAMMA_NORM else 1.0)
        ax_coh.set_title(f"COH_{tag}", fontsize=10)
        ax_coh.set_xticks([])
        ax_coh.set_yticks([])
        fig.colorbar(im_coh, ax=ax_coh, orientation='vertical', fraction=0.046, pad=0.02, label='Coherence')

<details>
<summary>9.5 Merge and plot backscatter (dB) mosaics (per date)</summary>

</details>

In [ ]:
def _load_water_mask_match(mask_path, shape, transform, crs):
    with rasterio.open(mask_path) as src:
        src_mask = src.read(1)
        if src.crs == crs and src.transform == transform and src_mask.shape == shape:
            return src_mask
        dst = np.zeros(shape, dtype=src_mask.dtype)
        reproject(
            source=src_mask,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.nearest,
        )
        return dst

def _apply_water_mask(arr, mask):
    # mask: 1 = keep land, 0 = water
    return np.where(mask == 0, np.nan, arr)


def _trim_nan_border(arr, transform):
    data = arr[0] if arr.ndim == 3 else arr
    mask = np.isfinite(data) & (data != 0)
    if not mask.any():
        return arr, transform
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1
    data = data[r0:r1, c0:c1]
    if arr.ndim == 3:
        arr = data[None, ...]
    else:
        arr = data
    new_transform = transform * Affine.translation(c0, r0)
    return arr, new_transform

# Build per-date backscatter (dB) mosaics directly from subset H5 (no per-burst GeoTIFFs)
os.makedirs(f"{savedir}/tifs", exist_ok=True)

date_tags = sorted(cslc_df.startTime.astype(str).str.replace('-', '').unique())

def _save_mosaic_utm(out_path, mosaic, transform, epsg):
    with rasterio.open(
        out_path, "w", driver="GTiff", height=mosaic.shape[0], width=mosaic.shape[1],
        count=1, dtype=mosaic.dtype, crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
    ) as dst_ds:
        dst_ds.write(mosaic, 1)

def _save_mosaic_utm_to_wgs84(out_path, mosaic, transform, epsg):
    dst_crs = "EPSG:4326"
    src_crs = CRS.from_epsg(epsg)
    height, width = mosaic.shape
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, width, height, *rasterio.transform.array_bounds(height, width, transform)
    )
    dst = np.empty((dst_height, dst_width), dtype=mosaic.dtype)
    reproject(
        source=mosaic,
        destination=dst,
        src_transform=transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
    )
    with rasterio.open(
        out_path, "w", driver="GTiff", height=dst_height, width=dst_width, count=1,
        dtype=dst.dtype, crs=dst_crs, transform=dst_transform, nodata=np.nan
    ) as dst_ds:
        dst_ds.write(dst, 1)

# Mosaicking helper (in memory)
def _mosaic_arrays(arrays, transforms, epsg):
    # Convert arrays to in-memory rasterio datasets via MemoryFile
    srcs = []
    for arr, transform in zip(arrays, transforms):
        mem = MemoryFile()
        ds = mem.open(
            driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1, dtype=arr.dtype,
            crs=CRS.from_epsg(epsg), transform=transform, nodata=np.nan
        )
        ds.write(arr, 1)
        srcs.append(ds)
    dest, out_transform = merge.merge(srcs, method=custom_merge)
    for ds in srcs:
        ds.close()
    return dest[0], out_transform


def _reproject_calibration_to_data_grid(sigma, cal_x, cal_y, cal_dx, cal_dy, xcoor, ycoor, dx, dy, epsg):
    cal_dy_signed = (cal_y[1] - cal_y[0]) if len(cal_y) > 1 else -cal_dy
    data_dy_signed = (ycoor[1] - ycoor[0]) if len(ycoor) > 1 else -dy
    if sigma.shape == (len(ycoor), len(xcoor)) and cal_dx == dx and abs(cal_dy_signed) == abs(data_dy_signed):
        return sigma
    src_transform = from_origin(cal_x[0], cal_y[0], cal_dx, abs(cal_dy_signed))
    dst_transform = from_origin(xcoor[0], ycoor[0], dx, abs(data_dy_signed))
    dst = np.empty((len(ycoor), len(xcoor)), dtype=sigma.dtype)
    reproject(
        source=sigma,
        destination=dst,
        src_transform=src_transform,
        src_crs=CRS.from_epsg(epsg),
        dst_transform=dst_transform,
        dst_crs=CRS.from_epsg(epsg),
        resampling=Resampling.bilinear,
    )
    return dst

looks_y, looks_x = MULTILOOK
# Backscatter calibration mode tag for filenames
bsc_mode_tag = CALIBRATION_MODE
beta_naught = None
for date_tag in date_tags:
    # collect subset H5 files for this date
    rows = cslc_df[cslc_df.startTime.astype(str).str.replace('-', '') == date_tag]
    arrays = []
    transforms = []
    epsg = None
    for fileID in rows.fileID:
        subset_path = f"{savedir}/subset_cslc/{fileID}.h5"
        with h5py.File(subset_path, 'r') as h5:
            cslc = h5['/data/VV'][:]
            xcoor = h5['/data/x_coordinates'][:]
            ycoor = h5['/data/y_coordinates'][:]
            dx = int(h5['/data/x_spacing'][()])
            dy = int(h5['/data/y_spacing'][()])
            epsg = int(h5['/data/projection'][()])
            sigma = h5['/metadata/calibration_information/sigma_naught'][:]
            cal_x = h5['/metadata/calibration_information/x_coordinates'][:]
            cal_y = h5['/metadata/calibration_information/y_coordinates'][:]
            cal_dx = int(h5['/metadata/calibration_information/x_spacing'][()])
            cal_dy = int(h5['/metadata/calibration_information/y_spacing'][()])
            beta_naught = h5['/metadata/calibration_information/beta_naught'][()]
        power_ml = _multilook(np.abs(cslc)**2, looks_y, looks_x)
        if CALIBRATION_MODE == 'sigma0':
            sigma_on_data = _reproject_calibration_to_data_grid(
                sigma, cal_x, cal_y, cal_dx, cal_dy, xcoor, ycoor, dx, dy, epsg
            )
            sigma_ml = _multilook(sigma_on_data, looks_y, looks_x)
            sigma_ml = np.where(sigma_ml <= 0, np.nan, sigma_ml)
            sigma_factor = sigma_ml**2 if CALIBRATION_FACTOR_IS_AMPLITUDE else sigma_ml
            bsc = 10*np.log10(power_ml / sigma_factor)
        elif CALIBRATION_MODE == 'beta0':
            beta = beta_naught
            if beta is None or beta <= 0:
                bsc = np.full(power_ml.shape, np.nan, dtype=power_ml.dtype)
            else:
                beta_factor = beta**2 if CALIBRATION_FACTOR_IS_AMPLITUDE else beta
                bsc = 10*np.log10(power_ml / beta_factor)
        else:
            raise ValueError("CALIBRATION_MODE must be 'sigma0' or 'beta0'")
        dy_signed = (ycoor[1] - ycoor[0]) if len(ycoor) > 1 else -dy
        x0 = xcoor[0] + (looks_x - 1) * dx / 2
        y0 = ycoor[0] + (looks_y - 1) * dy_signed / 2
        transform = from_origin(x0, y0, dx*looks_x, np.abs(dy_signed)*looks_y)
        arrays.append(bsc)
        transforms.append(transform)

    if not arrays:
        continue
    mosaic, out_transform = _mosaic_arrays(arrays, transforms, epsg)
    out_path_utm = f"{savedir}/tifs/merged_bsc_{bsc_mode_tag}_{date_tag}.tif"
    mosaic, out_transform = _trim_nan_border(mosaic, out_transform)
    if APPLY_WATER_MASK and WATER_MASK_PATH:
        mask = _load_water_mask_match(WATER_MASK_PATH, mosaic.shape, out_transform, CRS.from_epsg(epsg))
        mosaic = _apply_water_mask(mosaic, mask)
    if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path_utm)):
        _save_mosaic_utm(out_path_utm, mosaic, out_transform, epsg)
    if SAVE_WGS84:
        out_path_wgs84 = f"{savedir}/tifs/WGS84/merged_bsc_{bsc_mode_tag}_WGS84_{date_tag}.tif"
        if (not SKIP_EXISTING_TIFS) or (not os.path.exists(out_path_wgs84)):
            _save_mosaic_utm_to_wgs84(out_path_wgs84, mosaic, out_transform, epsg)

# Plot merged backscatter (dB) mosaics in a grid (native CRS from saved GeoTIFFs)

# Output dir for backscatter PNGs
bsc_png_dir = f"{savedir}/bsc_png"
os.makedirs(bsc_png_dir, exist_ok=True)

paths = sorted(glob.glob(f"{savedir}/tifs/merged_bsc_{bsc_mode_tag}_*.tif"))
paths = [p for p in paths if 'WGS84' not in p]
all_vals = []
for p in paths:
    da = rioxarray.open_rasterio(p)[0]
    all_vals.append(da.values.ravel())
if all_vals:
    all_vals = np.concatenate(all_vals)
    gmin = np.nanpercentile(all_vals, 2)
    gmax = np.nanpercentile(all_vals, 90)
else:
    gmin, gmax = None, None
n = len(paths)
if n == 0:
    print('No merged backscatter files found')
else:
    # Save ALL backscatter PNGs
    for path in paths:
        src = rioxarray.open_rasterio(path)
        bsc = src[0]
        minlon, minlat, maxlon, maxlat = bsc.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        fig, ax = plt.subplots(figsize=(5,4))
        im = ax.imshow(bsc.values, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=gmin, vmax=gmax)
        tag = path.split(f'merged_bsc_{bsc_mode_tag}_')[-1].replace('.tif','')
        ax.set_title(f"{tag}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.02, label=f"{bsc_mode_tag} (dB)")
        out_png = os.path.join(bsc_png_dir, f"bsc_{tag}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    # Show only last 5 in notebook
    display_paths = paths[-5:]
    n = len(display_paths)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), constrained_layout=True)
    axes = axes.ravel()
    for ax, path in zip(axes, display_paths):
        src = rioxarray.open_rasterio(path)
        bsc = src[0]
        minlon, minlat, maxlon, maxlat = bsc.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        im = ax.imshow(bsc.values, cmap='gray', interpolation='none', origin='upper', extent=bbox, vmin=gmin, vmax=gmax)
        tag = path.split(f'merged_bsc_{bsc_mode_tag}_')[-1].replace('.tif','')
        ax.set_title(f"{tag}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis('off')
    fig.colorbar(im, ax=axes.tolist(), orientation='vertical', fraction=0.02, pad=0.02, label=f"{bsc_mode_tag} (dB)")

<details>
<summary>10. Monthly mean coherence calendar (per year)</summary>

This calendar bins each pair into a month using the midpoint date between the reference and secondary scenes.
</details>

In [ ]:
# # CALENDAR_CMAP = ROCKET_CMAP  # previous
CALENDAR_CMAP = cmc.bilbao
# CALENDAR_CMAP = cmc.bilbao

# Build an index of merged coherence files by midpoint year-month
records = []
for path in sorted(glob.glob(f"{savedir}/tifs/merged_coh_*.tif")):
        tag = path.split('merged_coh_')[-1].replace('.tif','')
        try:
            ref_str, sec_str = tag.split('-')
            ref_date = pd.to_datetime(ref_str, format='%Y%m%d')
            sec_date = pd.to_datetime(sec_str, format='%Y%m%d')
            mid_date = ref_date + (sec_date - ref_date) / 2
        except Exception:
            continue
        records.append({"path": path, "mid_date": mid_date})

df_paths = pd.DataFrame(records)
if df_paths.empty:
    print('No merged coherence files found for calendar')
    raise SystemExit

# Apply current date range using midpoint date
date_start_day = dateStart.date()
date_end_day = dateEnd.date()
df_paths = df_paths[(df_paths['mid_date'].dt.date >= date_start_day) & (df_paths['mid_date'].dt.date <= date_end_day)]

# Calendar year labeling
if USE_WATER_YEAR:
    # Water year starts Oct (10) and ends Sep (9)
    df_paths['year'] = df_paths['mid_date'].dt.year + (df_paths['mid_date'].dt.month >= 10).astype(int)
    month_order = [10,11,12,1,2,3,4,5,6,7,8,9]
    month_labels = ['Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep']
else:
    df_paths['year'] = df_paths['mid_date'].dt.year
    month_order = list(range(1,13))
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

years = sorted(df_paths['year'].unique())

# Contrast stretch for low coherence (red = low)
norm = mcolors.PowerNorm(gamma=0.3, vmin=0, vmax=1)

# One row per year, 12 columns
fig, axes = plt.subplots(len(years), 12, figsize=(24, 2.5*len(years)), constrained_layout=True)
if len(years) == 1:
    axes = np.array([axes])

for row_idx, y in enumerate(years):
    # pick a template for consistent grid within the year (first available file)
    year_paths = df_paths[df_paths['year'] == y]['path'].tolist()
    if not year_paths:
        continue
    template = rioxarray.open_rasterio(year_paths[0])

    for col_idx, m in enumerate(month_order):
        ax = axes[row_idx, col_idx]
        month_paths = df_paths[(df_paths['year'] == y) & (df_paths['mid_date'].dt.month == m)]['path'].tolist()
        if USE_WATER_YEAR:
            year_for_month = y - 1 if m in (10, 11, 12) else y
        else:
            year_for_month = y
        title = f"{month_labels[col_idx]} {year_for_month}"
        if not month_paths:
            ax.set_title(title, fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            # keep a visible box for empty months
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(0.8)
                spine.set_color('0.5')
            continue
        stacks = []
        for p in month_paths:
            da = rioxarray.open_rasterio(p)
            da = da.rio.reproject_match(template)
            stacks.append(da)
        da_month = xr.concat(stacks, dim='stack').mean(dim='stack', skipna=True)
        minlon, minlat, maxlon, maxlat = da_month.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        im = ax.imshow(da_month.values.squeeze(), cmap=CALENDAR_CMAP, norm=norm, origin='upper', extent=bbox, interpolation='none')
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color('0.5')
    # left-side year label
    if USE_WATER_YEAR:
        label = f"WY {y}"
    else:
        label = str(y)
    axes[row_idx, 0].set_ylabel(label, rotation=90, labelpad=6, fontsize=9)
    axes[row_idx, 0].yaxis.set_label_coords(-0.06, 0.5)

fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.02, pad=0.02, label='Mean coherence')

<details>
<summary>11. Create GIF animations (Backscatter (dB) + Coherence)</summary>

</details>

In [ ]:
# GIF helper functions

def _global_bounds(paths):
    bounds = []
    for p in paths:
        da = rioxarray.open_rasterio(p)[0]
        minx, miny, maxx, maxy = da.rio.bounds()
        bounds.append((minx, miny, maxx, maxy))
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)
    return [minx, maxx, miny, maxy]

def _extract_date_from_tag(tag):
    m = re.search(r"(\d{8})", tag)
    if not m:
        return None
    try:
        return pd.to_datetime(m.group(1), format="%Y%m%d")
    except Exception:
        return None

def _format_title(tag, date=None):
    if date is not None:
        return date.strftime('%Y-%m-%d')
    m = re.findall(r"(\d{8})", tag)
    if len(m) >= 2:
        try:
            d0 = pd.to_datetime(m[0], format="%Y%m%d")
            d1 = pd.to_datetime(m[1], format="%Y%m%d")
            return f"{d0.strftime('%Y-%m-%d')} to {d1.strftime('%Y-%m-%d')}"
        except Exception:
            pass
    if len(m) == 1:
        try:
            d0 = pd.to_datetime(m[0], format="%Y%m%d")
            return d0.strftime('%Y-%m-%d')
        except Exception:
            pass
    return tag

def _filter_paths_by_date(paths, date_start, date_end):
    if not paths:
        return paths
    ds = pd.to_datetime(date_start).date()
    de = pd.to_datetime(date_end).date()
    kept = []
    for p in paths:
        tag = os.path.basename(p).replace('.tif','')
        m = re.findall(r"(\d{8})", tag)
        if len(m) >= 2:
            try:
                d0 = pd.to_datetime(m[0], format="%Y%m%d").date()
                d1 = pd.to_datetime(m[1], format="%Y%m%d").date()
                mid = d0 + (d1 - d0) / 2
                mid_date = mid if hasattr(mid, 'year') else mid.date()
                if ds <= mid_date <= de:
                    kept.append(p)
            except Exception:
                kept.append(p)
        elif len(m) == 1:
            try:
                d0 = pd.to_datetime(m[0], format="%Y%m%d").date()
                if ds <= d0 <= de:
                    kept.append(p)
            except Exception:
                kept.append(p)
        else:
            kept.append(p)
    return kept

def _render_frames(tif_paths, out_dir, cmap, vmin=None, vmax=None, title_prefix=None, extent=None, cbar_label=None, cbar_ticks=None, template=None, dates=None, title_dates=None, show_time_bar=False, norm=None):
    os.makedirs(out_dir, exist_ok=True)
    frames = []
    if template is None and tif_paths:
        template = rioxarray.open_rasterio(tif_paths[0])[0]
    if extent is None and template is not None:
        minlon, minlat, maxlon, maxlat = template.rio.bounds()
        extent = [minlon, maxlon, minlat, maxlat]

    date_min = date_max = None
    if show_time_bar and dates:
        valid_dates = [d for d in dates if d is not None]
        if valid_dates:
            date_min = min(valid_dates)
            date_max = max(valid_dates)

    for i, p in enumerate(tif_paths):
        da = rioxarray.open_rasterio(p)[0]
        if template is not None:
            da = da.rio.reproject_match(template)
        fig, ax = plt.subplots(figsize=(6,4))
        im = ax.imshow(da.values, cmap=cmap, origin='upper', extent=extent, norm=norm, vmin=None if norm is not None else vmin, vmax=None if norm is not None else vmax)
        tag = os.path.basename(p).replace('.tif','')
        if title_prefix is None:
            title_date = title_dates[i] if title_dates and i < len(title_dates) else None
            title = _format_title(tag, title_date)
        else:
            title = f"{title_prefix}{tag}"
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        if cbar_label:
            cb.set_label(cbar_label)
        if cbar_ticks is not None:
            cb.set_ticks(cbar_ticks)

        if show_time_bar and date_min is not None and date_max is not None:
            ax_time = fig.add_axes([0.12, 0.04, 0.76, 0.05])
            ax_time.plot([0, 1], [0.5, 0.5], color="0.6", linewidth=3, solid_capstyle='round')
            cur_date = dates[i] if dates and i < len(dates) else None
            if cur_date is not None and date_max != date_min:
                pos = (cur_date - date_min) / (date_max - date_min)
                pos = max(0, min(1, float(pos)))
            else:
                pos = 0.0 if date_max == date_min else 0.5
            ax_time.plot([pos, pos], [0.2, 0.8], color="crimson", linewidth=2, zorder=5)
            ax_time.scatter([pos], [0.5], s=90, color="crimson", zorder=6)

            if date_max == date_min:
                tick_dates = [date_min]
            else:
                tick_dates = pd.date_range(date_min, date_max, periods=5)
            for d in tick_dates:
                t = (d - date_min) / (date_max - date_min)
                t = max(0, min(1, float(t)))
                ax_time.plot([t, t], [0.35, 0.65], color="0.4", linewidth=1)
                ax_time.text(t, -0.05, d.strftime('%Y-%m-%d'), ha='center', va='top', fontsize=6)

            ax_time.set_xlim(0, 1)
            ax_time.set_ylim(0, 1)
            ax_time.axis('off')

        frame_path = os.path.join(out_dir, f"{tag}.png")
        fig.savefig(frame_path, dpi=150)
        plt.close(fig)
        frames.append(frame_path)
    return frames

def _pad_frames(frame_paths):
    imgs = [imageio.imread(f) for f in frame_paths]
    max_h = max(im.shape[0] for im in imgs)
    max_w = max(im.shape[1] for im in imgs)
    padded = []
    for im in imgs:
        pad_h = max_h - im.shape[0]
        pad_w = max_w - im.shape[1]
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        padded.append(np.pad(im, ((top, bottom), (left, right), (0, 0)), mode='edge'))
    return padded


In [ ]:
# Prepare coherence extent/template for GIFs
coh_template = None
coh_extent = None
if coh_paths:
    coh_template = rioxarray.open_rasterio(coh_paths[0])[0]
    minlon, minlat, maxlon, maxlat = coh_template.rio.bounds()
    coh_extent = [minlon, maxlon, minlat, maxlat]


In [ ]:
# GIF prep (paths, dates, extent)
gif_dir = f"{savedir}/gifs"
os.makedirs(gif_dir, exist_ok=True)

coh_paths = _filter_paths_by_date(coh_paths, dateStart, dateEnd)
coh_dates = []
for p in coh_paths:
    tag = Path(p).name.replace('merged_coh_', '').replace('.tif', '')
    try:
        ref_str, sec_str = tag.split('-')
        ref_date = pd.to_datetime(ref_str, format='%Y%m%d')
        sec_date = pd.to_datetime(sec_str, format='%Y%m%d')
        coh_dates.append(ref_date + (sec_date - ref_date) / 2)
    except Exception:
        coh_dates.append(None)

coh_template = None
coh_extent = None
if coh_paths:
    coh_template = rioxarray.open_rasterio(coh_paths[0])[0]
    minlon, minlat, maxlon, maxlat = coh_template.rio.bounds()
    coh_extent = [minlon, maxlon, minlat, maxlat]


In [ ]:
# Coherence GIF
# Adjust GIF speed based on number of frames

def _gif_duration(n_frames, min_sec=0.3, max_sec=1.2, target_total_sec=12.0):
    if n_frames <= 0:
        return max_sec
    return max(min_sec, min(max_sec, target_total_sec / n_frames))

coh_norm = mcolors.PowerNorm(gamma=COH_NORM_GAMMA, vmin=0, vmax=1) if COH_USE_GAMMA_NORM else None
if coh_paths:
    coh_frames = _render_frames(
        coh_paths, f"{gif_dir}/coh_frames", cmap=cmc.bilbao, vmin=0, vmax=1,
        title_prefix=None, extent=coh_extent, cbar_label='Coherence', cbar_ticks=[0,0.5,1],
        template=coh_template, dates=coh_dates, title_dates=None, show_time_bar=True, norm=coh_norm
    )
    coh_gif = f"{gif_dir}/coherence.gif"
    coh_imgs = _pad_frames(coh_frames)
    imageio.mimsave(coh_gif, coh_imgs, duration=_gif_duration(len(coh_imgs)))
    print(f"Wrote {coh_gif}")
else:
    print('No merged coherence files found for GIF')


# Monthly mean coherence GIF
if coh_paths:
    monthly_dir = f"{gif_dir}/coh_monthly_frames"
    os.makedirs(monthly_dir, exist_ok=True)
    # Build month index using midpoint dates
    monthly = {}
    for p, d in zip(coh_paths, coh_dates):
        if d is None:
            continue
        key = d.strftime('%Y-%m')
        monthly.setdefault(key, []).append(p)

    monthly_frames = []
    for key in sorted(monthly.keys()):
        stacks = []
        for p in monthly[key]:
            da = rioxarray.open_rasterio(p)[0]
            if coh_template is not None:
                da = da.rio.reproject_match(coh_template)
            stacks.append(da)
        if not stacks:
            continue
        da_month = xr.concat(stacks, dim='stack').mean(dim='stack', skipna=True)
        minlon, minlat, maxlon, maxlat = da_month.rio.bounds()
        bbox = [minlon, maxlon, minlat, maxlat]
        fig, ax = plt.subplots(figsize=(6,4))
        im = ax.imshow(da_month.values, cmap=cmc.bilbao, origin='upper', extent=bbox, interpolation='none', norm=coh_norm, vmin=None if COH_USE_GAMMA_NORM else 0, vmax=None if COH_USE_GAMMA_NORM else 1)
        ax.set_title(key, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cb.set_label('Mean coherence')
        frame_path = os.path.join(monthly_dir, f"{key}.png")
        fig.savefig(frame_path, dpi=150)
        plt.close(fig)
        monthly_frames.append(frame_path)

    if monthly_frames:
        monthly_gif = f"{gif_dir}/coherence_monthly.gif"
        monthly_imgs = _pad_frames(monthly_frames)
        imageio.mimsave(monthly_gif, monthly_imgs, duration=_gif_duration(len(monthly_imgs)))
        print(f"Wrote {monthly_gif}")
    else:
        print('No monthly coherence frames found for GIF')

# Backscatter GIF (sigma0/beta0)
bsc_paths = sorted(glob.glob(f"{savedir}/tifs/merged_bsc_{CALIBRATION_MODE}_*.tif"))
bsc_paths = _filter_paths_by_date(bsc_paths, dateStart, dateEnd)
bsc_dates = []

# Fixed color range across the series
bsc_all = []
for p in bsc_paths:
    da = rioxarray.open_rasterio(p)[0]
    bsc_all.append(da.values.ravel())
if bsc_all:
    bsc_all = np.concatenate(bsc_all)
    bsc_vmin = np.nanpercentile(bsc_all, 2)
    bsc_vmax = np.nanpercentile(bsc_all, 90)
else:
    bsc_vmin, bsc_vmax = None, None
for p in bsc_paths:
    tag = Path(p).name.replace('.tif','')
    bsc_dates.append(_extract_date_from_tag(tag))

if bsc_paths:
    bsc_template = rioxarray.open_rasterio(bsc_paths[0])[0]
    minlon, minlat, maxlon, maxlat = bsc_template.rio.bounds()
    bsc_extent = [minlon, maxlon, minlat, maxlat]
    bsc_frames = _render_frames(
        bsc_paths, f"{gif_dir}/bsc_frames_{CALIBRATION_MODE}", cmap=cmc.bilbao,
        vmin=bsc_vmin, vmax=bsc_vmax, title_prefix=None, extent=bsc_extent,
        cbar_label=f"Backscatter ({CALIBRATION_MODE}) dB", cbar_ticks=None,
        template=bsc_template, dates=bsc_dates, title_dates=None, show_time_bar=True
    )
    bsc_gif = f"{gif_dir}/backscatter_{CALIBRATION_MODE}.gif"
    bsc_imgs = _pad_frames(bsc_frames)
    imageio.mimsave(bsc_gif, bsc_imgs, duration=_gif_duration(len(bsc_imgs)))
    print(f"Wrote {bsc_gif}")
else:
    print('No merged backscatter files found for GIF')
